In [1]:
"""
Copyright (C) <2025>  <The Ohio State University>

This program is free software: you can redistribute it and/or modify it under
the terms of the GNU General Public License as published by the Free Software
Foundation, either version 3 of the License, or (at your option) any later version.
This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
See the GNU General Public License for more details. You should have received a copy of the
GNU General Public License along with this program.  If not, see <https://www.gnu.org/licenses/>

This script analyzes the output of RNA folding simulations to quantify the effect
of natural indels at arbitrary binding sites in human transcripts on the change
in free energy (∆∆G) of RNA structures. It is structurally very similar to
indels_analysis.py but operates on a different dataset.

The script calculates the mean, variance, and standard deviation of ∆∆G values
across all sequences. It performs these calculations globally (all data) and
for subsets of data grouped by indel size.
""";

In [2]:
%reset -f
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
import math
from scipy.stats import bootstrap
import os

In [4]:
working_directory = "/content/drive/MyDrive/work/data/RNA-protein/IndelPolymorphismsModulateDistalRNAProteinInteractions"
os.chdir(working_directory)

In [5]:
# --- Script Parameters ---
seq_length = 150
footprint = 7

# Defines the indel sizes to be analyzed
omit_list = [i+1 for i in range(5)]
iter_variable = 0

In [6]:
# the data files that can be processed by this notebook
file_names = ['dataHumanNoBS/varySites/varySites150',
              'dataHuman/varySites/varySites150',
              'dataHuman/varySites_5prime/varySites_5prime150']

# the dataset currently being processed
file_name = file_names[2]

# Output files for standard deviation, confidence intervals, variance, and mean
infilename = file_name + '.dat'
infile = open(infilename, 'r')

dev_outfilename = file_name +'_dev.dat'
dev_outfile = open(dev_outfilename, 'w')

ci_outfilename = file_name + '_dev_ci.dat'
ci_outfile = open(ci_outfilename, 'w')

var_outfilename = file_name +'_var.dat'
var_outfile = open(var_outfilename, 'w')

avg_outfilename = file_name +'_avg.dat'
avg_outfile = open(avg_outfilename, 'w')

In [7]:
# --- Write Headers to Output Files ---
dev_outfile.write("# position global" + "".join([" "+str(i+1)+"dels" for i in range(5)]) + "\n")
var_outfile.write("# position global" + "".join([" "+str(i+1)+"dels" for i in range(5)]) + "\n")
avg_outfile.write("# position global" + "".join([" "+str(i+1)+"dels" for i in range(5)]) + "\n")
ci_outfile.write("# position global_ci_low global_ci_high" + "".join([" "+str(i+1)+"dels_ci_low "+str(i+1)+"dels_ci_high" for i in range(5)]) + "\n")

175

In [8]:
# --- Helper function to calculate bootstrap CI for standard deviation ---
def get_stdev_ci(data):
    """
    Calculates the 95% bootstrap confidence interval for the standard deviation.

    Args:
        data (list or np.array): A list of numerical data.

    Returns:
        tuple: A tuple containing the lower and upper bounds of the confidence
               interval.
    """

    # Ensure data is a numpy array for the bootstrap function
    data_arr = np.array(data)

    # The bootstrap function requires samples to be passed as a tuple, so we pass (data_arr,).
    res = bootstrap((data_arr,),
                    np.std,
                    confidence_level=0.95,
                    random_state=1,
                    n_resamples=5000)#batch=batch_size)

    return (res.confidence_interval.low, res.confidence_interval.high)

In [9]:
# --- Data Loading and Processing ---

# Read the first line to determine number of columns and positions
first_line = infile.readline()
nts_positions = first_line.split()[1:len(first_line.split())-1]
num_cols = len(first_line.split())-1

# Initialize data structures
data_semi_global = [[[] for i in range(len(omit_list))] for j in range(num_cols-1)]
data_global = [[] for j in range(num_cols-1)]

# Reset file pointer to the beginning
infile.seek(0)

0

In [10]:
# --- Group data by indel size (semi-global) ---
for i in omit_list:
  infile.seek(0)
  for line in infile:
    if line == "\n":
      continue
    fields = line.split()
    if fields[0] != '#':
      if int(fields[num_cols-1]) == i:
        for k in range(len(fields)-1):
          data_semi_global[k][iter_variable].append(float(fields[k]))
  iter_variable += 1

# Reset file pointer
infile.seek(0)

0

In [11]:
# --- Load all data (global) ---
for line in infile:
  if line == "\n":
    continue
  fields = line.split()
  if fields[0] != '#':
    for k in range(len(fields)-1):
      data_global[k].append(float(fields[k]))

# Reset file pointer
infile.seek(0)

0

In [12]:
# --- Subsample large datasets for consistent statistics ---
subsample_size = 20000
for i in range(len(data_global)):
    if len(data_global[i]) > subsample_size:
        np.random.seed(1)
        data_global[i] = np.random.choice(data_global[i], size=subsample_size, replace=False)

for i in range(len(data_semi_global)):
    for j in range(len(data_semi_global[i])):
        if len(data_semi_global[i][j]) > subsample_size:
            np.random.seed(1)
            data_semi_global[i][j] = np.random.choice(data_semi_global[i][j], size=subsample_size, replace=False)

In [23]:
# Create a mapping between column index and nucleotide position
mapping = [(i, nts_positions[i]) for i in range(len(data_global))]

# --- Calculate and Write Statistics ---
for x, y in mapping:
    # Write standard deviation, variance, and mean
    dev_outfile.write( str(y) + " " + str(math.sqrt(np.var(data_global[x])))+ "".join([" "+str(math.sqrt(np.var(data_semi_global[x][i]))) for i in range(len(omit_list))]) +" \n")
    var_outfile.write( str(y) + " " + str(np.var(data_global[x]))+ "".join([" "+str(np.var(data_semi_global[x][i])) for i in range(len(omit_list))]) +" \n")
    avg_outfile.write( str(y) + " " + str(np.mean(data_global[x]))+ "".join([" "+str(np.mean(data_semi_global[x][i])) for i in range(len(omit_list))]) +" \n")

    # --- Calculate and write confidence intervals ---
    global_ci = get_stdev_ci(data_global[x])
    semiglobal_cis = [get_stdev_ci(data_semi_global[x][i]) for i in range(len(omit_list))]

    # Format the CI string for writing
    ci_string = " ".join([f"{ci[0]} {ci[1]}" for ci in semiglobal_cis])

    # Write the formatted CI data to the new CI file
    ci_outfile.write(f"{y} {global_ci[0]} {global_ci[1]} {ci_string}\n")

In [24]:
# --- Close all file handles ---
infile.close()
dev_outfile.close()
var_outfile.close()
avg_outfile.close()
ci_outfile.close()